# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.datasets import  fetch_california_housing
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
df_clean = spaceship.dropna()
display(df_clean.isna().sum())


df_clean.loc[:, 'Cabin'] = df_clean['Cabin'].str[0]
print(sorted(df_clean['Cabin'].unique()))


df_clean = df_clean.drop(columns=["PassengerId", "Name"]) 

df_final = pd.get_dummies(df_clean, drop_first=True) 

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'T']


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
# 1. Define the target
y = df_final['Transported'].astype(int)

# 2. Define the features (everything except the target)
X = df_final.drop(columns=['Transported'])

**Perform Train Test Split**

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled= scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [ ]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# 1. Initialize the four ensemble models
bagging_model  = BaggingClassifier(random_state=42)
pasting_model  = BaggingClassifier(bootstrap=False, random_state=42) # bootstrap=False makes it Pasting
rf_model       = RandomForestClassifier(random_state=42)
ada_model      = AdaBoostClassifier(random_state=42)
gb_model       = GradientBoostingClassifier(random_state=42)

# Create a dictionary to loop through them cleanly
ensemble_models = {
    "Bagging (With Replacement)": bagging_model,
    "Pasting (Without Replacement)": pasting_model,
    "Random Forest": rf_model,
    "AdaBoost": ada_model,
    "Gradient Boosting": gb_model
}

# 2. Train and evaluate each model automatically
print("--- Ensemble Methods Performance ---")
for name, model in ensemble_models.items():
    # Fit the model on raw, unscaled training data
    model.fit(X_train, y_train)
    
    # Generate predictions on raw test data
    y_pred = model.predict(X_test)
    
    # Calculate accuracy
    acc = accuracy_score(y_test, y_pred)
    print(f"{name:<30} Accuracy: {acc:.2%}")

- Bagging and Pasting

Bootstrap Aggregating: trains multiple models in parallel on random subsets of your data with replacement (meaning the same row can be picked twice). They all vote on the final answert.Pasting is the exact same thing, but without replacement. 

In [15]:
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Bagging model
bagging_model = BaggingClassifier(random_state=42)

# Train the model
bagging_model.fit(X_train, y_train)

# Predict and Evaluate
y_pred_bagging = bagging_model.predict(X_test)
print("--- Bagging Classifier Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_bagging):.2%}\n")
print(classification_report(y_test, y_pred_bagging))

--- Bagging Classifier Performance ---
Accuracy: 79.20%

              precision    recall  f1-score   support

           0       0.78      0.81      0.79       653
           1       0.81      0.77      0.79       669

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



In [ ]:
#Pasting(Without Replacement)--> every random subset contains entirely unique rows
#from your training pool. We achieve this by setting bootstrap=False

from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Pasting model (bootstrap=False turns off replacement)
pasting_model = BaggingClassifier(bootstrap=False, random_state=42)

# Train the model
pasting_model.fit(X_train, y_train)

# Predict and Evaluate
y_pred_pasting = pasting_model.predict(X_test)
print("--- Pasting Classifier Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_pasting):.2%}\n")
print(classification_report(y_test, y_pred_pasting))

- Random Forests

An advanced extension of Bagging.It trains hundreds of trees in parallel,but adds extra randomness by forcing each tree to only look at a random substed of features (columns). This stops the trees from copying each other and builds a highly diverse "forest". 

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Random Forest model
rf_model = RandomForestClassifier(random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

# Predict and Evaluate
y_pred_rf = rf_model.predict(X_test)
print("--- Random Forest Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.2%}\n")
print(classification_report(y_test, y_pred_rf))

--- Random Forest Performance ---
Accuracy: 80.64%

              precision    recall  f1-score   support

           0       0.79      0.82      0.81       653
           1       0.82      0.79      0.81       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



- Gradient Boosting

Another sequentional boosting method. Instead of tweaking passenger weights like AdaBoost, it trains the next tree to directly predict the residuals (the exact mathematical errors) left behind by the previous tree.

**How it starts**: It starts with a baseline guess, which is just the overall global average of the target column. 

**The Correction step**: It calculates the exact mathematical difference(the residual) between that baseline guess and the actual true values

**The Next Tree**: Is not trained to predict if a passenger was transported (1 or 0). Instead, it is trained to predict those residual errors.

**The Optimization**: It keeps repeating this, finding the path of the steepest descent(the Gradient) to minimize the overall error. Every new tree is added directly to the formula to substract a bit more of the remaining error. 

The learning rate (often called shrinkage or $\alpha$ / learning_rate in Scikit-Learn) is a multiplier between 0.0 and 1.0. 

A very common standard default is 0.1.Instead of trusting Tree 1 completely, Gradient Boosting says: "I hear your suggestion of +10, Tree 1. 

But I'm only going to take 10% of your advice right now."

Let's look at the math layout:

New Prediction = Current Prediction + ($\alpha$ x Tree Prediction)

New Prediction = 0 + (0.1 x 10) = +1

In [18]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

#Initialize the AdaBoost model
ada_model= AdaBoostClassifier(random_state=42)

#train the model
ada_model.fit(X_train, y_train)

#Predict and Evaluate
y_pred_ada = ada_model.predict(X_test)
print("---AdaBoost Performance ---")
print(f"Accuracy:{accuracy_score(y_test, y_pred_ada): .2%}\n")
print(classification_report(y_test, y_pred_ada))

---AdaBoost Performance ---
Accuracy: 77.69%

              precision    recall  f1-score   support

           0       0.74      0.85      0.79       653
           1       0.83      0.71      0.76       669

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322



- Adaptive Boosting

AdaBoost: trains models sequentially (one after the other). The first tree makes guesses, and the second tree deliberalitely focuses its atttention on the passengers the first tree got wrong. It keeps repeating this, passing a heavy focus down the chain to fix mistakes.

**Difference to gradient boosting**: fixes mistakes by shiftig its focus onto the individual data rows it got wrong. It does this by assigning a physical weight to every row in your dataset. 
**How it starts**: Every passenger row starts with equal weight. The first tree tries
to predict who was transported.
**The Correction step**: AdaBoost looks at the passengers the first tree guessed incorrectly. It increases the weight of those specific incorrect rows and lowers the weight of the correct ones. 

**The Next Tree**: When the second tree is trained, it sees those heavily weighted rows as "more important". It modifies its splits specifically to get those tricky rows right. 

**The Final Voting**: when making a final prediction, the trees don't have equal say. Trees that performed exceptionally well on the training data get a louder, heavier vote than trees that struggled.

**Summary comparison** 

1.AdaBoost

2.Gradient Boosting 

**How it fixes errors**
1. By increasing the weight of misclassified data rows.
2. By training the next tree to predict the remaining mathematical errors (residuals)

**Tree structure**
1. Usually uses "stumps" (trees with a depth of only 1 split)
2. Uses larger, shallow trees (typically a depth of 3 to 6 splits)

**Final Voting**
1. Each tree is assigned a specific voting power weight based on its accuracy.
2. All trees are added together equally, but scaled down by a learning rate (alpha) to prevent overfitting.

**Sensitivity**
1. Highly sensitive to outliers because it will aggressively over-weight a single weird row.
2. More robust to outliers because it minimizes a global loss function smoothly.

In [19]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Gradient Boosting model
gb_model = GradientBoostingClassifier(random_state=42)

# Train the model
gb_model.fit(X_train, y_train)

# Predict and Evaluate
y_pred_gb = gb_model.predict(X_test)
print("--- Gradient Boosting Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_gb):.2%}\n")
print(classification_report(y_test, y_pred_gb))

--- Gradient Boosting Performance ---
Accuracy: 80.71%

              precision    recall  f1-score   support

           0       0.84      0.76      0.79       653
           1       0.78      0.86      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



Which model is the best and why?

Gradient Boosting is the best model.

1. **Tabular Domination**: Gradient Boosting excels at capturing complex, non-linear interactions between mixed data types (like matching a passenger's numerical Age with their specific categorical Cabin deck).




2. **Sequential Error Correction**: Unlike Random Forests or Bagging which build trees in isolation, Gradient Boosting explicitly forces every new tree to learn from and minimize the exact mathematical mistakes (residuals) of the previous tree.




3. **Robustness to Feature Outliers**: By relying on a shallow tree structure and scaling down each tree's impact via a cautious learning rate ($\alpha = 0.1$), it smoothly navigates extreme variance in passenger spending features without letting anomalies warp the model's trajectory.